# Motor: ResNet18 CE vs Poly-1 CE (development comparison)

**Run all cells top-to-bottom with a GPU runtime.** This notebook uses the live GitHub repository, not an uploaded ZIP. It will not invent performance scores. The 99 historical images are previously inspected and therefore cannot be called a fresh independent test.


In [ ]:
import subprocess, pathlib, os
repo = pathlib.Path('/content/DL-Computer-vision')
if not repo.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/ID9999999999999/DL-Computer-vision.git',str(repo)], check=True)
project=repo/'COURSEWORK_NAZIH'
assert (project/'compare_poly1.py').exists(), 'Coursework folder missing from GitHub'
os.chdir(project)
print('Project:', project)


## 1 — Install and verify

The historical model weights are not stored in GitHub, so the optional checkpoint-reload test will be skipped unless you place the checkpoint at `resnet18_supervised/best_model.pt`. All other unit tests run without it.


In [ ]:
subprocess.run(['python','-m','pip','install','-q','-r','requirements.txt'],check=True)
subprocess.run(['python','offline_preflight.py'],check=True)
subprocess.run(['python','-m','unittest','test_pipeline.py','-v'],check=True)


## 2 — Fetch original images

The downloader retrieves only the manifest's 99 historical images with ZIP byte ranges; the provider may change its range API. The process stops if requests or SHA256 hashes do not match. It does not open any fresh blind-test set.


In [ ]:
subprocess.run(['python','download_missing.py','--workers','4'],check=True)


## 3 — Controlled training comparison (GPU recommended)

This retrains **both** models from the same initialization, using identical train/validation images and budget; only the loss changes.


In [ ]:
import torch
print('GPU:',torch.cuda.is_available())
subprocess.run(['python','compare_poly1.py','--mode','both','--epsilon','1.0','--device','auto'],check=True)


## 4 — Read actual results (not independent test results)


In [ ]:
import json
from pathlib import Path
out=Path('comparison_results')
for method in ['ce','poly1']:
    z=json.loads((out/f'{method}_result.json').read_text())
    print(method, z['best_validation'])
print('Difference:',json.loads((out/'comparison.json').read_text()))
print('Reminder: development validation only. Physical specimen independence not yet confirmed.')


## 5 — Export all outputs

Download the actual results ZIP for your report. If you want to publish small JSON/CSV results to GitHub, add them only after reviewing their validity and limitations. Never upload weights or raw M2AD images.


In [ ]:
import shutil
from google.colab import files
archive=shutil.make_archive('/content/Nazih_Motor_Development_Results','zip',root_dir=str(project),base_dir='comparison_results')
files.download(archive)
